# Predict Genre for Multiple Unseen Books
Loads the trained classifiers and applies them to up to 5 books the models have never seen.

**Run order:** `BOW.ipynb` → `Book partitioner.ipynb` → `Genre classifier.ipynb` → **this notebook**

**Required files (produced by prior notebooks):**
- `FeatureTrainingData/vectorizer.pkl`
- `model_logistic_regression.pkl`
- `model_svm.pkl`
- `model_random_forest.pkl`
- `model_naive_bayes.pkl`
- `model_xgboost.pkl`
- `model_distilbert/`

In [33]:
import re
import os
import random
import joblib
import numpy as np
import pandas as pd
from collections import Counter

## Configuration
Add up to 5 books to the `BOOKS` list.
- Set `"path"` to the book's `.txt` file path.
- Set `"genre"` to the true genre (e.g. `"fantasy"`) if you want accuracy reported, or `None` if unknown.

In [ ]:
BOOKS = [
    {"path": "KnownBooks/PlagueShip.txt",         "genre": "sci-fi"},
    {"path": "KnownBooks/LifeOfKingEdwardVII.txt", "genre": "biography"},
    {"path": "KnownBooks/RomeoAndJuliet.txt",      "genre": "romance"},
    {"path": "KnownBooks/AliceInWonderland.txt",   "genre": "fantasy"},
    {"path": "KnownBooks/CallOfCthulu.txt",        "genre": "horror"},
]

PARTITION_SIZE = 400    # words per partition (must match training)
NUM_PARTITIONS = 200    # how many random partitions to sample
RANDOM_STATE   = 42

GENRES = ["biography", "fantasy", "horror", "romance", "sci-fi"]

## Load Saved Models

In [ ]:
from xgb_wrapper import XGBStringClassifier      # needed to deserialise model_xgboost.pkl
from bert_wrapper import DistilBertGenreClassifier

vec      = joblib.load('FeatureTrainingData/vectorizer.pkl')
lr       = joblib.load('models/model_logistic_regression.pkl')
svm      = joblib.load('models/model_svm.pkl')
rf       = joblib.load('models/model_random_forest.pkl')
nb       = joblib.load('models/model_naive_bayes.pkl')
xgb_clf  = joblib.load('models/model_xgboost.pkl')
bert_clf = DistilBertGenreClassifier('models/model_distilbert')

print(f"Vectorizer vocabulary size : {len(vec.vocabulary_)} words")
print("Models loaded: Logistic Regression, SVM, Random Forest, Naive Bayes, XGBoost, DistilBERT")

## Helper Functions
Defines functions to clean, partition, and predict genre for a single book.
Strips Project Gutenberg header/footer boilerplate (same logic as `BOW.ipynb`).

In [ ]:
START_PATTERN = re.compile(r'\*{3}\s*START OF THE PROJECT GUTENBERG EBOOK[^\n]*\*{3}', re.IGNORECASE)
END_PATTERN   = re.compile(r'\*{3}\s*END OF THE PROJECT GUTENBERG EBOOK[^\n]*\*{3}',   re.IGNORECASE)

def clean_book(path):
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        raw = f.read()
    start_match = START_PATTERN.search(raw)
    text = raw[start_match.end():] if start_match else raw
    end_match = END_PATTERN.search(text)
    text = text[:end_match.start()] if end_match else text
    return text.strip().split()

def sample_partitions(tokens, n=NUM_PARTITIONS, size=PARTITION_SIZE, seed=RANDOM_STATE):
    random.seed(seed)
    max_start = len(tokens) - size
    if max_start < n:
        raise ValueError(f"Book too short: {len(tokens)} words, need at least {n + size}.")
    starts = sorted(random.sample(range(max_start), n))
    return [' '.join(tokens[s : s + size]) for s in starts]

MODEL_KEYS = [
    ('LR',         'preds_lr'),
    ('SVM',        'preds_svm'),
    ('RF',         'preds_rf'),
    ('NB',         'preds_nb'),
    ('XGBoost',    'preds_xgb'),
    ('DistilBERT', 'preds_bert'),
]

def predict_book(path, known_genre=None):
    name   = os.path.splitext(os.path.basename(path))[0]
    tokens = clean_book(path)
    parts  = sample_partitions(tokens)
    X      = vec.transform(parts)
    return {
        'name':        name,
        'genre':       known_genre,
        'preds_lr':    lr.predict(X),
        'preds_svm':   svm.predict(X),
        'preds_rf':    rf.predict(X),
        'preds_nb':    nb.predict(X),
        'preds_xgb':   xgb_clf.predict(X),
        'preds_bert':  bert_clf.predict(parts),   # raw text — not TF-IDF
        'probs_lr':    lr.predict_proba(X),
        'word_count':  len(tokens),
        'n_parts':     len(parts),
    }

print("Helper functions defined.")

## Process All Books
Runs the full pipeline (clean → partition → vectorise → predict) for each book in `BOOKS`.

In [37]:
book_results = []
for entry in BOOKS:
    path, genre = entry['path'], entry['genre']
    name = os.path.splitext(os.path.basename(path))[0]
    print(f"Processing: {name} ...", end=' ')
    try:
        result = predict_book(path, known_genre=genre)
        book_results.append(result)
        print(f"done  ({result['word_count']:,} words, {result['n_parts']} partitions)")
    except Exception as e:
        print(f"FAILED — {e}")

print(f"\n{len(book_results)}/{len(BOOKS)} books processed successfully.")

Processing: PlagueShip ... done  (60,036 words, 200 partitions)
Processing: LifeOfKingEdwardVII ... done  (147,715 words, 200 partitions)
Processing: RomeoAndJuliet ... done  (25,958 words, 200 partitions)
Processing: AliceInWonderland ... done  (26,525 words, 200 partitions)
Processing: CallOfCthulu ... done  (11,968 words, 200 partitions)

5/5 books processed successfully.


## Per-Book Results
Majority-vote across all partitions to produce a single genre prediction per book.

In [ ]:
def majority_vote(predictions):
    return Counter(predictions).most_common(1)[0][0]

for r in book_results:
    print("=" * 55)
    print(f"  Book: {r['name']}")
    print("=" * 55)
    for label, key in MODEL_KEYS:
        vote = majority_vote(r[key])
        if r['genre']:
            acc = (r[key] == r['genre']).mean()
            print(f"  {label:10} → {vote:15} (partition acc: {acc:.1%})")
        else:
            print(f"  {label:10} → {vote}")
    if r['genre']:
        print(f"  True genre : {r['genre']}")

    vote_table = pd.DataFrame(
        {label: pd.Series(r[key]).value_counts().reindex(GENRES, fill_value=0)
         for label, key in MODEL_KEYS}
    )
    print()
    print(vote_table.to_string())
    print()

## Comparison Summary
Single table comparing all books side by side, plus overall accuracy for any labelled books.

In [ ]:
rows = []
for r in book_results:
    row = {'Book': r['name'], 'True Genre': r['genre'] if r['genre'] else '—'}
    for label, key in MODEL_KEYS:
        vote = majority_vote(r[key])
        row[label] = vote
        if r['genre']:
            row[f'{label} ✓'] = '✓' if vote == r['genre'] else '✗'
        else:
            row[f'{label} ✓'] = '—'
    row['LR Confidence'] = f"{r['probs_lr'].max(axis=1).mean():.1%}"
    rows.append(row)

comparison_df = pd.DataFrame(rows).set_index('Book')
print("=== COMPARISON SUMMARY ===\n")
display(comparison_df)

labelled = [r for r in book_results if r['genre']]
if labelled:
    print(f"\nOverall book-level accuracy ({len(labelled)} labelled books):")
    for label, key in MODEL_KEYS:
        acc = np.mean([majority_vote(r[key]) == r['genre'] for r in labelled])
        print(f"  {label:10}: {acc:.1%}")

## Vote Distribution Charts
One row of bar charts per book (Logistic Regression + SVM side by side).

In [ ]:
import matplotlib.pyplot as plt

MODEL_COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
x     = np.arange(len(GENRES))
width = 0.13

n_books = len(book_results)
fig, axes = plt.subplots(n_books, 1, figsize=(12, 4 * n_books))
if n_books == 1:
    axes = [axes]

for ax, r in zip(axes, book_results):
    for i, (label, key) in enumerate(MODEL_KEYS):
        counts = pd.Series(r[key]).value_counts().reindex(GENRES, fill_value=0)
        ax.bar(x + (i - 2.5) * width, counts.values, width,
               label=label, color=MODEL_COLORS[i], alpha=0.85)
    genre_label = f"  (true: {r['genre']})" if r['genre'] else ""
    ax.set_title(f"{r['name']}{genre_label}", fontsize=11)
    ax.set_xticks(x)
    ax.set_xticklabels(GENRES)
    ax.set_ylabel('Partitions')
    ax.set_ylim(0, NUM_PARTITIONS + 30)
    ax.legend(loc='upper right', fontsize=8, ncol=6)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('images/multi_book_genre_votes.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: images/multi_book_genre_votes.png')